### Import all necessary modules and libraries

In [ ]:
from agent.agent import BaseAgent
from env.replay import Backtest

import numpy as np
import pandas as pd
import datetime

### Define your agent by inheriting from BaseAgent

The agent should act according to the task.

1. Opening a position
- Collect and update minimum and maximum price of the security
- Check if (1) current exposure is zero and (2) current drawdown / upswing exceeds given threshold
- Enter a long position / Enter a short position

2. Closing a position with profit
- Calculate the current relative unrealized profit of your position
- If the relative profit exceeds given threshold 
- Close the position

3. Closing a position before the trading session ends  
- If we reach the end of the trading day, e.g. 4:15 PM
- Close the position & do not submit orders anymore


In [ ]:
class AdvancedAgent(BaseAgent):

    def __init__(self, name: str, barrier_open: float,
                 barrier_close: float, quantity: int):
        """
        Trading agent implementation example. Improved version.

        :param name:
            str, agent name
        :param barrier_open:
            float, if barrier is hit, agent opens a position
        :param barrier_close:
            float, if barrier is hit, agent closes a position with profit
        :param stop_loss:
            float, if stop loss is hit, agent closes a position with loss
        :param quantity:
            int, defines the amount of shares that are traded
        """

        super(AdvancedAgent, self).__init__(name)

        # static attributes from arguments
        self.quantity = quantity
        self.barrier_open = barrier_open
        self.barrier_close = barrier_close

        # further static attributes
        self.end_time = datetime.time(16, 15)
        self.market_interface.transaction_cost_factor = 0

        # dynamic attributes
        self.trading_phase = False  # indicates whether algo is ready to trade
        self.max_price = np.NaN  # initialize max price of stock
        self.min_price = np.NaN  # initialize max price of stock

    def on_quote(self, market_id:str, book_state:pd.Series):
        """
        This method is called after a new quote.

        :param market_id:
            str, market identifier
        :param book_state:
            pd.Series, including timestamp, bid/ask price/size for 10 levels
        """
        if self.trading_phase:

            # Calculate midpoint
            midpoint = (book_state['L1-BidPrice'] + book_state['L1-AskPrice']) / 2
            midpoint = np.round(midpoint, 4)
            # alternative: midpoint = self.markets[market_id].mid_point

            # Update min and max prices of episode
            self.max_price = max(midpoint, self.max_price)
            self.min_price = min(midpoint, self.min_price)

            # Calculate current drawdown & upswing
            drawdown = min((midpoint/self.max_price - 1), 0)
            upswing = max((midpoint/self.min_price - 1), 0)

            # Conditions for opening a position
            # (1) no position is open
            # (2) no order is waiting for execution
            if not self.market_interface.exposure[market_id] and \
               not self.market_interface.get_filtered_orders(
                   market_id, status="ACTIVE"):

                # Submit market buy order if drawdown exceeds barrier
                if abs(drawdown) > self.barrier_open:
                    self.market_interface.submit_order(
                        market_id, "buy", self.quantity)

                # Submit market sell order if upswing exceeds barrier
                elif abs(upswing) > self.barrier_open:
                    self.market_interface.submit_order(
                        market_id, "sell", self.quantity)

        # Conditions for closing long position
        # (1) Long position exists for this market
        if self.market_interface.exposure[market_id] > 0:
            # Get price of trade
            exec_price = self.market_interface.get_filtered_trades(market_id)[-1].price
            trade_profit = midpoint/exec_price - 1

            # Close long position if
            # (2) barrier_close is hit
            if trade_profit > self.barrier_close:
                self.market_interface.submit_order(market_id, "sell", self.quantity)
                self.max_price = midpoint  # reset current max

        # Conditions to for closing short position
        # (1) Short position exists for this market
        elif self.market_interface.exposure[market_id] < 0:
            # Get price of trade
            exec_price = self.market_interface.get_filtered_trades(market_id)[-1].price
            trade_profit = -1 * (midpoint/exec_price - 1)

            # Close short position if
            # (2) barrier_close is hit
            if trade_profit > self.barrier_close:
                self.market_interface.submit_order(market_id, "buy", self.quantity)
                self.min_price = midpoint  # reset current min


    def on_trade(self, market_id:str, trades_state:pd.Series):
        """
        This method is called after a new trade.

        :param market_id:
            str, market identifier
        :param trades_state:
            pd.Series, including timestamp, price, quantity
        """
        pass

    def on_time(self, timestamp:pd.Timestamp, timestamp_next:pd.Timestamp):
        """
        This method is called with every iteration and provides the timestamps
        for both current and next iteration. The given interval may be used to
        submit orders before a specific point in time.

        :param timestamp:
            pd.Timestamp, timestamp recorded
        :param timestamp_next:
            pd.Timestamp, timestamp recorded in next iteration
        """

        # Enter trading phase if
        # (1) current time < end_time
        # (2) trading_phase is False up to now
        if timestamp.time() < self.end_time and not self.trading_phase:
            print('Algo is now able to trade...')
            self.trading_phase = True

        # Close trading phase if
        # (1) current time > end_time
        # (2) trading_phase is True up to now
        elif timestamp.time() > self.end_time and self.trading_phase:

            for market_id in self.market_interface.market_state_list.keys():

                # cancel active orders for this market
                [self.market_interface.cancel_order(order) for order in
                 self.market_interface.get_filtered_orders(
                     market_id, status="ACTIVE")]

                # close positions for this market
                if self.market_interface.exposure[market_id] > 0:
                    self.market_interface.submit_order(
                        market_id, "sell", self.quantity)
                if self.market_interface.exposure[market_id] < 0:
                    self.market_interface.submit_order(
                        market_id, "buy", self.quantity)
            # Set bool to False
            self.trading_phase = False


### Define your agent and backtest

1. Define an instance of your agent by specifying the parameters defined in the constructor.
2. Define an instance of the Backtest class by passing your agent instance.
3. Define the list of asset identifiers you want to trade on.

In [ ]:
agent = AdvancedAgent(
    name="AdvancedAgent1",
    barrier_open=0.0005,
    barrier_close=0.0005,
    quantity=20,
)

backtest = Backtest(
    agent=agent, 
)

identifier_list = [
    # Aktie1
    "Aktie1.BOOK", "Aktie1.TRADES",
]

### Run your backtest

There are several methods to run your backtest. Here, we show two options:

1. Run your agent within predefined time periods (called "episodes") using the method **run_episode_list**. The episode list requires three date-time strings per episode: the first defines the start of the episode buffer. The simulation starts here to fill the order book based on historical order flow. The second timestamp is the start of the backtest where your agent can interact with the market. The third timestamp represents the end of the backtest.
2. Run your agent against a series of randomly selected episodes using the method **run_episode_generator**. The backtest periods are selected within the given date range between date_start and date_end. The episode_interval dividies each day into intervals of the specified length (in minutes). From these intervals, episodes are randomly selected. Each episode consists of: buffer period defined by episode_buffer (in minutes) and backtest period defined by episode_length (in minutes). Thus, the buffer and length of each episode should sum up to the episode_interval. To ensure reproducibility, you can set a random seed. You can also specify the number of episodes to run. If episode_shuffle is set to True, the episodes are randomly shuffled before running the backtest.

Furthermore, both methods require the source directory where the historical market data is stored. Ultimately, you can define the sampling frequency (in events) to define how often your agent can interact with the market. By default, this is set to 1, meaning that your agent can react to every market event (quote or trade). Higher values imply that your agent can only react every n-th event, but this can speed up the backtesting process.

In [ ]:
# Option 1: run agent against a single episode defined by start and end date
backtest.run_episode_list(
    identifier_list=identifier_list,
    source_directory=r'ITFM_Data',
    episode_list=[
        ("2025-03-17T08:00:00", "2025-03-17T08:05:00", "2025-03-17T08:35:00"),
        ("2025-03-17T09:00:00", "2025-03-17T09:05:00", "2025-03-17T09:35:00"),
        ("2025-03-17T10:00:00", "2025-03-17T10:05:00", "2025-03-17T10:35:00"),
        # ...
    ],
)

To keep the runtime of this example short, the second option is commented out.

In [ ]:
# Option 2: run agent against a series of generated episodes, that is,
# generate episodes with the same episode_buffer and episode_length
#backtest.run_episode_generator(
#    identifier_list=identifier_list,
#    source_directory=r"ITFM_Data",
#    date_start="2025-03-17",
#    date_end="2025-03-17",
#    episode_interval=20,
#    episode_shuffle=False,
#    episode_buffer=5,
#    episode_length=15,
#    num_episodes=2,
#    seed=123,
#    sampling_freq=5,
#)

### Evaluate your backtesting results

You can access the backtesting results via the attribute **results** of your backtest instance. This attribute is a list of dictionaries, where each dictionary contains various a list of orders, trades and performance metrics for each episode in the backtest. You can analyze these results to evaluate the performance of your agent.

In [ ]:
# TODO: EVALUATE YOUR BACKTESTING RESULTS
# E.g., you can use result list backtest.results to analyze your runs
pnl_realized_all_runs = [d['PnL_realized'] for d in backtest.results]
pnl_realized_all_runs = pd.DataFrame(pnl_realized_all_runs)
pnl_realized_all_runs.to_csv('PnL_realized_all_runs.csv', index_label='Run')

In [ ]:
backtest.results